# Freddie Mac Performance Target Development

## Purpose

This notebook develops a leakage-safe loan-level credit outcome from the 2015 Freddie Mac monthly performance sample. It follows the same configuration, validation, display, and persistence pattern used in Notebook 03.

The notebook does not treat a modeling choice as a Freddie Mac rule. It separately measures:

- **90+ day serious delinquency:** numeric delinquency status 3 or higher, plus `RA` for REO acquisition;
- **terminal credit events:** Zero Balance Codes `02`, `03`, `09`, and `15`; and
- **composite credit outcome:** either of the two conditions above.

Candidate 12-, 24-, and 36-month windows are compared before the 24-month primary target is saved. The notebook reports both overall sample retention and follow-up coverage among loans observable from month 0. The 24-month label is retained only when observable-cohort coverage meets the configured threshold. The other horizons remain available for sensitivity analysis.

### Documentation basis

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf)
- [Freddie Mac Non-Standard Dataset Summary Statistics](https://www.freddiemac.com/fmac-resources/research/pdf/nonstandard_dataset_summary_statistics.pdf)

Freddie Mac defines delinquency status `3` as 90-119 days delinquent and `RA` as REO acquisition. Its published performance summaries group Zero Balance Codes `02`, `03`, `09`, and `15` as non-prepayment credit events. The 24-month horizon and 95% observable-cohort coverage threshold are research-design choices, not Freddie Mac requirements.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_performance import (
    CREDIT_EVENT_ZERO_BALANCE_CODES,
    DOCUMENTED_ZERO_BALANCE_CODES,
    PERFORMANCE_COLUMNS,
    TARGET_PERFORMANCE_COLUMNS,
    create_horizon_targets,
    load_performance,
    prepare_performance_timeline,
    summarize_horizon_targets,
    validate_performance_for_target,
)


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

Notebook directory: e:\capstone_project\mortgage-model-reliability\notebooks
Project root: e:\capstone_project\mortgage-model-reliability


## 1. File Configuration

Define the analysis year, candidate horizons, observable-cohort coverage requirement, and source paths once. The coverage requirement is applied after excluding left-truncated loans because their histories do not begin by month 0 and they are not targetable without inventing unobserved outcomes.

The official Release 47 sample performance filename is `sample_svcg_2015.txt`; the alternate `sample_perf_2015.txt` path is checked only to support the previously extracted local filename.

The cleaned origination Parquet produced by Notebook 03 supplies `FIRST PAYMENT DATE` and `ORIGINATION COHORT`.

In [2]:
YEAR = 2015
HORIZONS = (12, 24, 36)
SELECTED_HORIZON = 24
MINIMUM_COVERAGE_PERCENTAGE = 95.0

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)

PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

PERFORMANCE_FILE_CANDIDATES = [
    (
        RAW_DATA_DIRECTORY
        / f"sample_{YEAR}"
        / f"sample_svcg_{YEAR}.txt"
    ),
    (
        RAW_DATA_DIRECTORY
        / f"sample_{YEAR}"
        / f"sample_perf_{YEAR}.txt"
    ),
]

ORIGINATION_FILE = (
    PROCESSED_DATA_DIRECTORY
    / f"origination_{YEAR}_clean.parquet"
)

file_check = pd.DataFrame(
    [
        {
            "file_type": "performance_candidate",
            "exists": file_path.exists(),
            "path": str(file_path),
        }
        for file_path in PERFORMANCE_FILE_CANDIDATES
    ]
    + [
        {
            "file_type": "cleaned_origination",
            "exists": ORIGINATION_FILE.exists(),
            "path": str(ORIGINATION_FILE),
        }
    ]
)

display(file_check)

existing_performance_files = [
    file_path
    for file_path in PERFORMANCE_FILE_CANDIDATES
    if file_path.exists()
]

if not existing_performance_files:
    raise FileNotFoundError(
        "No 2015 performance file was found in the expected locations."
    )

if not ORIGINATION_FILE.exists():
    raise FileNotFoundError(
        "The cleaned 2015 origination Parquet is missing. "
        "Run Notebook 03 first."
    )

PERFORMANCE_FILE = existing_performance_files[0]

print("Selected performance file:", PERFORMANCE_FILE)
print("Selected origination file:", ORIGINATION_FILE)

,file_type,exists,path
0,performance_candidate,False,e:\capstone_project\mortgage-model-reliability...
1,performance_candidate,True,e:\capstone_project\mortgage-model-reliability...
2,cleaned_origination,True,e:\capstone_project\mortgage-model-reliability...


Selected performance file: e:\capstone_project\mortgage-model-reliability\data\raw\Freddie_Mac\sample_2015\sample_perf_2015.txt
Selected origination file: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_clean.parquet


## 2. Confirm the Expected 35-Field Schema

The current Freddie Mac monthly performance layout contains 35 pipe-delimited fields. This display records every expected field position and identifies the five fields required for target construction.

All 35 fields are structurally validated, but only the five target-relevant fields are loaded into memory. This avoids loading millions of rows across columns that are not needed to define the outcome.

In [3]:
schema_table = pd.DataFrame(
    {
        "position": range(
            1,
            len(PERFORMANCE_COLUMNS) + 1,
        ),
        "column": PERFORMANCE_COLUMNS,
    }
)

schema_table["target_relevant"] = (
    schema_table["column"].isin(
        TARGET_PERFORMANCE_COLUMNS
    )
)

display(schema_table)
print("Expected performance fields:", len(PERFORMANCE_COLUMNS))
print("Target-relevant fields:", len(TARGET_PERFORMANCE_COLUMNS))

,position,column,target_relevant
0,1,LOAN IDENTIFIER,True
1,2,PERIOD,True
2,3,CURRENT ACTUAL UPB,False
3,4,CURRENT LOAN DELINQUENCY STATUS,True
4,5,LOAN AGE,False
5,6,REMAINING MONTHS TO LEGAL MATURITY,False
6,7,UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT...,False
7,8,MODIFICATION FLAG,False
8,9,ZERO BALANCE CODE,True
9,10,ZERO BALANCE EFFECTIVE DATE,True


Expected performance fields: 35
Target-relevant fields: 5


## 3. Structurally Validate and Load the 2015 Data

Before pandas parsing, every raw row is scanned to confirm that it contains exactly 35 fields. The loader then reads only the five target-relevant performance columns while preserving identifiers, reporting periods, delinquency statuses, and zero-balance codes as strings.

Keeping code fields as strings prevents values such as `00`, `01`, and `09` from losing their documented representation.

In [4]:
performance_2015 = load_performance(
    file_path=PERFORMANCE_FILE,
    usecols=TARGET_PERFORMANCE_COLUMNS,
)

structure_validation = performance_2015.attrs[
    "structure_validation"
]

origination_2015 = pd.read_parquet(
    ORIGINATION_FILE,
    columns=[
        "LOAN IDENTIFIER",
        "FIRST PAYMENT DATE",
        "ORIGINATION COHORT",
    ],
)

structure_summary = pd.DataFrame(
    [
        {
            "year": YEAR,
            **structure_validation,
            "loaded_rows": performance_2015.shape[0],
            "loaded_columns": performance_2015.shape[1],
            "origination_loans": len(origination_2015),
        }
    ]
)

display(structure_summary)
display(performance_2015.head())

,year,rows_scanned,expected_fields_per_row,field_count_errors,loaded_rows,loaded_columns,origination_loans
0,2015,3467166,35,0,3467166,5,50000


,LOAN IDENTIFIER,PERIOD,CURRENT LOAN DELINQUENCY STATUS,ZERO BALANCE CODE,ZERO BALANCE EFFECTIVE DATE
0,F15Q10000025,201503,00,<NA>,<NA>
1,F15Q10000025,201504,00,<NA>,<NA>
2,F15Q10000025,201505,00,<NA>,<NA>
3,F15Q10000025,201506,00,<NA>,<NA>
4,F15Q10000025,201507,00,<NA>,<NA>


## 4. Validate Keys, Reporting Periods, and Documented Codes

Target construction requires one record per loan and monthly reporting period. This validation checks:

1. missing loan identifiers;
2. invalid `YYYYMM` reporting periods;
3. duplicate loan-period records;
4. undocumented delinquency-status representations;
5. undocumented Zero Balance Codes;
6. performance loans absent from the origination file; and
7. origination loans without performance records.

The first six conditions stop execution because they can make the label ambiguous or attach outcomes to the wrong loan. Origination loans without performance are reported and later marked target-ineligible rather than silently labeled zero.

In [5]:
validation_report = validate_performance_for_target(
    performance=performance_2015,
    origination=origination_2015,
)

validation_summary = pd.DataFrame(
    [
        {
            "year": YEAR,
            **validation_report,
        }
    ]
)

display(validation_summary)

,year,performance_rows,performance_columns_loaded,performance_loans,origination_loans,missing_loan_ids,invalid_periods,duplicate_loan_periods,unexpected_delinquency_statuses,unexpected_zero_balance_codes,unmatched_performance_loans,origination_loans_without_performance
0,2015,3467166,5,50000,50000,0,0,0,0,0,0,0


## 5. Profile Delinquency Statuses

Freddie Mac defines the delinquency field using the Mortgage Bankers Association method:

- `0` or `00`: current or less than 30 days delinquent;
- `1`: 30-59 days delinquent;
- `2`: 60-89 days delinquent;
- `3`: 90-119 days delinquent, with higher numeric values continuing in monthly increments; and
- `RA`: REO acquisition.

The primary serious-delinquency threshold begins at numeric status 3 because it is the documented 90-day boundary. Statuses 1 and 2 are earlier delinquency stages and are not positive labels for this target.

In [6]:
delinquency_profile = (
    performance_2015.assign(
        DELINQUENCY_STATUS=(
            performance_2015[
                "CURRENT LOAN DELINQUENCY STATUS"
            ]
            .astype("string")
            .str.strip()
            .fillna("<MISSING>")
        )
    )
    .groupby(
        "DELINQUENCY_STATUS",
        dropna=False,
        observed=True,
    )
    .agg(
        performance_rows=(
            "LOAN IDENTIFIER",
            "size",
        ),
        unique_loans=(
            "LOAN IDENTIFIER",
            "nunique",
        ),
    )
    .reset_index()
)

delinquency_profile["row_percentage"] = (
    delinquency_profile["performance_rows"]
    / len(performance_2015)
    * 100
)

display(
    delinquency_profile.sort_values(
        "DELINQUENCY_STATUS"
    ).reset_index(drop=True)
)

,DELINQUENCY_STATUS,performance_rows,unique_loans,row_percentage
0,00,3415433,50000,98.507917
1,01,23522,6626,0.678422
2,02,6686,2707,0.192838
3,03,3756,2042,0.108331
4,04,2597,1648,0.074903
...,...,...,...,...
67,67,1,1,0.000029
68,68,1,1,0.000029
69,69,1,1,0.000029
70,70,1,1,0.000029


## 6. Profile Termination Events

A Zero Balance Code explains why a loan became inactive:

- `01`: prepaid or matured - a voluntary payoff, not a credit event;
- `02`: third-party sale;
- `03`: short sale or charge-off;
- `09`: REO disposition;
- `15`: whole-loan sale;
- `16`: reperforming-loan securitization; and
- `96`: defect prior to property disposition.

For consistency with Freddie Mac's published performance summaries, codes `02`, `03`, `09`, and `15` form the terminal-credit-event component. Codes `01`, `16`, and `96` do not automatically create a positive label. A loan carrying one of those codes can still be positive if it previously reached 90+ delinquency.

In [7]:
zero_balance_profile = (
    performance_2015.assign(
        ZERO_BALANCE_CODE=(
            performance_2015["ZERO BALANCE CODE"]
            .astype("string")
            .str.strip()
            .str.zfill(2)
            .fillna("<ACTIVE>")
        )
    )
    .groupby(
        "ZERO_BALANCE_CODE",
        dropna=False,
        observed=True,
    )
    .agg(
        performance_rows=(
            "LOAN IDENTIFIER",
            "size",
        ),
        unique_loans=(
            "LOAN IDENTIFIER",
            "nunique",
        ),
    )
    .reset_index()
)

zero_balance_profile["documented_code"] = (
    zero_balance_profile["ZERO_BALANCE_CODE"].isin(
        DOCUMENTED_ZERO_BALANCE_CODES
    )
    | zero_balance_profile[
        "ZERO_BALANCE_CODE"
    ].eq("<ACTIVE>")
)

zero_balance_profile["credit_event_code"] = (
    zero_balance_profile["ZERO_BALANCE_CODE"].isin(
        CREDIT_EVENT_ZERO_BALANCE_CODES
    )
)

display(
    zero_balance_profile.sort_values(
        "ZERO_BALANCE_CODE"
    ).reset_index(drop=True)
)

,ZERO_BALANCE_CODE,performance_rows,unique_loans,documented_code,credit_event_code
0,01,38212,38212,True,False
1,02,45,45,True,True
2,03,18,18,True,True
3,09,55,55,True,True
4,15,18,18,True,True
5,16,108,108,True,False
6,96,35,35,True,False
7,<ACTIVE>,3428675,49976,True,False


## 7. Create the First-Payment-Relative Timeline

The observation clock is derived from `FIRST PAYMENT DATE` and each monthly `PERIOD`. Month 0 is the first scheduled payment month, month 1 is the next calendar month, and so on.

The raw `LOAN AGE` field is not used as the horizon clock because Freddie Mac documents that it resets after a loan modification. Using the calendar difference from the original first payment date keeps the outcome window stable and prevents modified loans from receiving a restarted target window.

Four additive event fields are created without modifying the raw columns:

- `SERIOUS DELINQUENCY EVENT`;
- `TERMINAL CREDIT EVENT`;
- `TERMINATION EVENT`; and
- `COMPOSITE CREDIT EVENT`.

In [8]:
performance_timeline_2015 = (
    prepare_performance_timeline(
        performance=performance_2015,
        origination=origination_2015,
    )
)

timeline_columns = [
    "LOAN IDENTIFIER",
    "FIRST PAYMENT DATE",
    "PERIOD",
    "MONTHS FROM FIRST PAYMENT",
    "CURRENT LOAN DELINQUENCY STATUS",
    "ZERO BALANCE CODE CLEAN",
    "SERIOUS DELINQUENCY EVENT",
    "TERMINAL CREDIT EVENT",
    "COMPOSITE CREDIT EVENT",
]

display(
    performance_timeline_2015[
        timeline_columns
    ].head(10)
)

,LOAN IDENTIFIER,FIRST PAYMENT DATE,PERIOD,MONTHS FROM FIRST PAYMENT,CURRENT LOAN DELINQUENCY STATUS,ZERO BALANCE CODE CLEAN,SERIOUS DELINQUENCY EVENT,TERMINAL CREDIT EVENT,COMPOSITE CREDIT EVENT
0,F15Q10000025,201504,201503,-1,00,<NA>,False,False,False
1,F15Q10000025,201504,201504,0,00,<NA>,False,False,False
2,F15Q10000025,201504,201505,1,00,<NA>,False,False,False
3,F15Q10000025,201504,201506,2,00,<NA>,False,False,False
4,F15Q10000025,201504,201507,3,00,<NA>,False,False,False
5,F15Q10000025,201504,201508,4,00,<NA>,False,False,False
6,F15Q10000025,201504,201509,5,00,<NA>,False,False,False
7,F15Q10000025,201504,201510,6,00,<NA>,False,False,False
8,F15Q10000025,201504,201511,7,00,<NA>,False,False,False
9,F15Q10000025,201504,201512,8,00,<NA>,False,False,False


## 8. Diagnose Acquisition Timing and Monthly Gaps

Freddie Mac performance history begins at acquisition, which may be later than origination or the first scheduled payment. A loan whose first observed record occurs after month 0 is left-truncated because an earlier credit event could be unobserved.

Monthly gaps are measured but are not automatically treated as target failures. Freddie Mac documents that gaps can occur because of settlement validation, reconciliation, or late servicing information. Eligibility therefore depends on the beginning and end of observable follow-up, while the number of internal gaps remains an audit field.

In [9]:
loan_observation = (
    performance_timeline_2015.groupby(
        "LOAN IDENTIFIER",
        observed=True,
    )
    .agg(
        first_observed_month=(
            "MONTHS FROM FIRST PAYMENT",
            "min",
        ),
        last_observed_month=(
            "MONTHS FROM FIRST PAYMENT",
            "max",
        ),
        observed_months=(
            "MONTHS FROM FIRST PAYMENT",
            "nunique",
        ),
    )
    .reset_index()
)

loan_observation["calendar_span_months"] = (
    loan_observation["last_observed_month"]
    - loan_observation["first_observed_month"]
    + 1
)

loan_observation["internal_missing_months"] = (
    loan_observation["calendar_span_months"]
    - loan_observation["observed_months"]
)

observation_diagnostics = pd.DataFrame(
    [
        {
            "loans_with_performance": len(
                loan_observation
            ),
            "left_truncated_loans": int(
                loan_observation[
                    "first_observed_month"
                ].gt(0).sum()
            ),
            "loans_with_internal_gaps": int(
                loan_observation[
                    "internal_missing_months"
                ].gt(0).sum()
            ),
            "minimum_first_observed_month": int(
                loan_observation[
                    "first_observed_month"
                ].min()
            ),
            "maximum_first_observed_month": int(
                loan_observation[
                    "first_observed_month"
                ].max()
            ),
            "minimum_last_observed_month": int(
                loan_observation[
                    "last_observed_month"
                ].min()
            ),
            "maximum_last_observed_month": int(
                loan_observation[
                    "last_observed_month"
                ].max()
            ),
        }
    ]
)

display(observation_diagnostics)
display(
    loan_observation[
        [
            "first_observed_month",
            "last_observed_month",
            "observed_months",
            "internal_missing_months",
        ]
    ].describe()
)

,loans_with_performance,left_truncated_loans,loans_with_internal_gaps,minimum_first_observed_month,maximum_first_observed_month,minimum_last_observed_month,maximum_last_observed_month
0,50000,2736,2,-2,59,-2,133


,first_observed_month,last_observed_month,observed_months,internal_missing_months
count,50000.00000,50000.000000,50000.000000,50000.00000
mean,0.34276,68.686140,69.343320,0.00006
std,7.15766,40.012239,40.254482,0.01000
min,-2.00000,-2.000000,1.000000,0.00000
25%,-1.00000,37.000000,36.000000,0.00000
50%,-1.00000,63.000000,64.000000,0.00000
75%,-1.00000,115.000000,112.000000,0.00000
max,59.00000,133.000000,135.000000,2.00000


## 9. Compare 12-, 24-, and 36-Month Horizons

For each horizon, a loan is eligible only when:

1. its performance history begins no later than month 0; and
2. it is observed through the final month of the horizon or reaches a documented termination event within the horizon.

This rule prevents a short, still-active performance history from being mislabeled as a negative outcome. A loan that terminates inside the window has a known outcome through termination and therefore has complete follow-up for this binary endpoint.

The comparison reports two distinct coverage measures:

- **sample retention percentage:** eligible loans divided by all sampled loans; and
- **observable-cohort coverage percentage:** eligible loans divided by loans whose history begins no later than month 0.

Left-truncated loans remain explicitly reported but are excluded from the second denominator because their earlier outcomes are unavailable. The 95% selection guard uses observable-cohort coverage to measure right-censoring among loans that could actually receive a reliable label.

The comparison also reports both the 90+ delinquency-only rate and the broader composite rate. This shows the exact incremental effect of including terminal credit events.

In [ ]:
target_candidates_2015 = create_horizon_targets(
    timeline=performance_timeline_2015,
    origination=origination_2015,
    horizons=HORIZONS,
)

horizon_summary_2015 = summarize_horizon_targets(
    target_candidates_2015
)

rate_columns = [
    "serious_delinquency_rate",
    "composite_credit_event_rate",
]

horizon_summary_display = (
    horizon_summary_2015.copy()
)

for column in rate_columns:
    horizon_summary_display[
        f"{column}_percentage"
    ] = (
        horizon_summary_display[column]
        * 100
    )

display(
    horizon_summary_display.drop(
        columns=rate_columns
    ).round(3)
)

## 10. Visualize Coverage and Event-Rate Tradeoffs

The left panel compares follow-up coverage among loans observable from month 0. This is the measure used by the 95% selection guard. Overall sample retention remains available in the summary table and is not hidden.

The right panel compares the serious-delinquency-only and composite event rates.

A longer horizon can capture more events, but it delays label availability and may reduce complete follow-up. The 24-month window is intended to balance these considerations; the figure makes that tradeoff visible rather than assuming it.

In [ ]:
figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12, 4.5),
)

axes[0].bar(
    horizon_summary_2015["horizon_months"].astype(str),
    horizon_summary_2015[
        "observable_cohort_coverage_percentage"
    ],
    color="#4C78A8",
)
axes[0].axhline(
    MINIMUM_COVERAGE_PERCENTAGE,
    color="#E45756",
    linestyle="--",
    label=(
        f"Minimum coverage "
        f"({MINIMUM_COVERAGE_PERCENTAGE:.0f}%)"
    ),
)
axes[0].set_title(
    "Observable-cohort follow-up coverage"
)
axes[0].set_xlabel("Horizon in months")
axes[0].set_ylabel("Coverage (%)")
axes[0].legend()

axes[1].plot(
    horizon_summary_2015["horizon_months"],
    (
        horizon_summary_2015[
            "serious_delinquency_rate"
        ]
        * 100
    ),
    marker="o",
    label="90+ delinquency",
)
axes[1].plot(
    horizon_summary_2015["horizon_months"],
    (
        horizon_summary_2015[
            "composite_credit_event_rate"
        ]
        * 100
    ),
    marker="o",
    label="Composite credit event",
)
axes[1].set_title("Eligible-loan event rates")
axes[1].set_xlabel("Horizon in months")
axes[1].set_ylabel("Event rate (%)")
axes[1].legend()

figure.suptitle(
    "2015 Freddie Mac Target-Horizon Comparison"
)
figure.tight_layout()
plt.show()

## 11. Quantify the Composite Target Components

The composite is not a vague grouping. This table separates eligible 24-month loans into four mutually exclusive categories:

1. 90+ delinquency without a terminal credit code;
2. terminal credit code without an observed 90+ status;
3. both components; and
4. neither component.

The terminal-only category shows exactly how many positive outcomes would be missed by using monthly delinquency status alone.

In [ ]:
selected_horizon_data = (
    target_candidates_2015.loc[
        target_candidates_2015[
            "HORIZON_MONTHS"
        ].eq(SELECTED_HORIZON)
        & target_candidates_2015[
            "TARGET_ELIGIBLE"
        ]
    ]
    .copy()
)

serious_event = selected_horizon_data[
    "SERIOUS_DELINQUENCY_WITHIN_HORIZON"
]
terminal_event = selected_horizon_data[
    "TERMINAL_CREDIT_EVENT_WITHIN_HORIZON"
]

component_summary = pd.DataFrame(
    [
        {
            "component": "90+ delinquency only",
            "loans": int(
                (serious_event & ~terminal_event).sum()
            ),
        },
        {
            "component": "Terminal credit event only",
            "loans": int(
                (~serious_event & terminal_event).sum()
            ),
        },
        {
            "component": "Both components",
            "loans": int(
                (serious_event & terminal_event).sum()
            ),
        },
        {
            "component": "Neither component",
            "loans": int(
                (~serious_event & ~terminal_event).sum()
            ),
        },
    ]
)

component_summary["percentage"] = (
    component_summary["loans"]
    / len(selected_horizon_data)
    * 100
)

display(component_summary.round(3))

## 12. Check the 24-Month Target by Origination Cohort

Quarterly event counts are required before modeling because a pooled annual rate can hide outcome differences across `2015Q1` through `2015Q4`.

This is a descriptive target check, not yet a drift test. Formal feature drift, outcome drift, discrimination, and calibration analysis occur in later stages.

In [ ]:
cohort_target_summary_2015 = (
    selected_horizon_data.groupby(
        "ORIGINATION COHORT",
        observed=True,
    )
    .agg(
        eligible_loans=(
            "LOAN IDENTIFIER",
            "size",
        ),
        serious_delinquency_events=(
            "TARGET_90_PLUS",
            "sum",
        ),
        serious_delinquency_rate=(
            "TARGET_90_PLUS",
            "mean",
        ),
        composite_credit_events=(
            "TARGET_COMPOSITE_CREDIT_EVENT",
            "sum",
        ),
        composite_credit_event_rate=(
            "TARGET_COMPOSITE_CREDIT_EVENT",
            "mean",
        ),
    )
    .reset_index()
)

cohort_target_summary_2015[
    "serious_delinquency_rate_percentage"
] = (
    cohort_target_summary_2015[
        "serious_delinquency_rate"
    ]
    * 100
)

cohort_target_summary_2015[
    "composite_credit_event_rate_percentage"
] = (
    cohort_target_summary_2015[
        "composite_credit_event_rate"
    ]
    * 100
)

display(
    cohort_target_summary_2015.drop(
        columns=[
            "serious_delinquency_rate",
            "composite_credit_event_rate",
        ]
    ).round(3)
)

## 13. Apply the 24-Month Selection Guard

The 24-month horizon is the primary project window because it supports an early-warning framing while allowing more time for serious outcomes to emerge than a 12-month label. A 36-month horizon is retained as a sensitivity analysis but would delay label availability by another year.

The notebook keeps overall sample retention and observable-cohort coverage separate:

- overall retention measures how much of the original 50,000-loan sample receives a label;
- observable-cohort coverage measures complete follow-up after excluding loans whose histories begin after month 0.

The code does not save the primary target unless 24-month observable-cohort coverage reaches 95%. If that condition fails, the notebook stops and the horizon decision must be revisited rather than silently accepting a heavily censored label.

In [ ]:
selected_horizon_summary = (
    horizon_summary_2015.loc[
        horizon_summary_2015[
            "horizon_months"
        ].eq(SELECTED_HORIZON)
    ]
    .iloc[0]
)

selected_observable_coverage = float(
    selected_horizon_summary[
        "observable_cohort_coverage_percentage"
    ]
)

selected_sample_retention = float(
    selected_horizon_summary[
        "sample_retention_percentage"
    ]
)

if selected_observable_coverage < MINIMUM_COVERAGE_PERCENTAGE:
    raise ValueError(
        f"{SELECTED_HORIZON}-month observable-cohort coverage "
        f"is {selected_observable_coverage:.3f}%, below the required "
        f"{MINIMUM_COVERAGE_PERCENTAGE:.3f}%."
    )

performance_target_2015 = (
    selected_horizon_data[
        [
            "LOAN IDENTIFIER",
            "ORIGINATION COHORT",
            "HORIZON_MONTHS",
            "TARGET_90_PLUS",
            "TARGET_COMPOSITE_CREDIT_EVENT",
            "FIRST_SERIOUS_DELINQUENCY_PERIOD",
            "FIRST_TERMINAL_CREDIT_EVENT_PERIOD",
            "FIRST_OBSERVED_MONTH",
            "LAST_OBSERVED_MONTH",
            "OBSERVED_MONTHS",
        ]
    ]
    .sort_values("LOAN IDENTIFIER")
    .reset_index(drop=True)
)

print(
    "Selected horizon:",
    SELECTED_HORIZON,
    "months",
)
print(
    "Overall sample retention:",
    f"{selected_sample_retention:.3f}%",
)
print(
    "Observable-cohort coverage:",
    f"{selected_observable_coverage:.3f}%",
)
print(
    "Eligible loans:",
    len(performance_target_2015),
)
print(
    "Composite positive loans:",
    int(
        performance_target_2015[
            "TARGET_COMPOSITE_CREDIT_EVENT"
        ].sum()
    ),
)
print(
    "Composite event rate:",
    (
        performance_target_2015[
            "TARGET_COMPOSITE_CREDIT_EVENT"
        ].mean()
        * 100
    ),
)

## 14. Save Auditable Target Outputs

Four outputs are saved:

1. all loan-level candidates for 12, 24, and 36 months;
2. the eligible 24-month modeling target;
3. the horizon-comparison summary; and
4. a JSON record containing the validation results and final target definition.

The validation JSON records both overall sample retention and observable-cohort coverage so the left-truncation exclusion cannot be mistaken for insufficient follow-up.

The saved final target retains the delinquency-only label and the composite label. This preserves transparency and allows later sensitivity analysis without rebuilding the monthly history.

In [ ]:
PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATHS = {
    "candidates": (
        PROCESSED_DATA_DIRECTORY
        / f"performance_target_candidates_{YEAR}.parquet"
    ),
    "selected_target": (
        PROCESSED_DATA_DIRECTORY
        / (
            f"performance_target_{YEAR}_"
            f"{SELECTED_HORIZON}m.parquet"
        )
    ),
    "horizon_summary": (
        PROCESSED_DATA_DIRECTORY
        / f"performance_target_{YEAR}_horizon_summary.csv"
    ),
    "validation": (
        PROCESSED_DATA_DIRECTORY
        / f"performance_target_{YEAR}_validation.json"
    ),
}

target_candidates_2015.to_parquet(
    OUTPUT_PATHS["candidates"],
    index=False,
    engine="pyarrow",
)

performance_target_2015.to_parquet(
    OUTPUT_PATHS["selected_target"],
    index=False,
    engine="pyarrow",
)

horizon_summary_2015.to_csv(
    OUTPUT_PATHS["horizon_summary"],
    index=False,
)

validation_payload = {
    "year": YEAR,
    "performance_file": str(PERFORMANCE_FILE),
    "selected_horizon_months": SELECTED_HORIZON,
    "minimum_coverage_percentage": (
        MINIMUM_COVERAGE_PERCENTAGE
    ),
    "selected_sample_retention_percentage": (
        selected_sample_retention
    ),
    "selected_observable_cohort_coverage_percentage": (
        selected_observable_coverage
    ),
    "target_definition": (
        "Numeric delinquency status >= 3 or RA, "
        "or Zero Balance Code in 02, 03, 09, 15, "
        "within months 0 through 23 from FIRST PAYMENT DATE."
    ),
    "structure_validation": structure_validation,
    "data_validation": validation_report,
    "eligible_loans": len(performance_target_2015),
    "serious_delinquency_events": int(
        performance_target_2015[
            "TARGET_90_PLUS"
        ].sum()
    ),
    "composite_credit_events": int(
        performance_target_2015[
            "TARGET_COMPOSITE_CREDIT_EVENT"
        ].sum()
    ),
}

with OUTPUT_PATHS["validation"].open(
    mode="w",
    encoding="utf-8",
) as validation_file:
    json.dump(
        validation_payload,
        validation_file,
        indent=2,
    )

for output_name, output_path in OUTPUT_PATHS.items():
    print(f"{output_name}: {output_path}")

## 15. Read Back and Validate the Saved Files

Persistence is part of the pipeline, so the notebook reopens every saved analytical table. It verifies:

- one candidate row per loan and horizon;
- only the configured horizons are present;
- one selected-target row per eligible loan;
- the selected horizon is 24 months;
- the final labels contain only 0 and 1; and
- saved summary counts agree with the in-memory results.

A successful write alone is not treated as proof that the saved artifact is correct.

In [ ]:
saved_candidates = pd.read_parquet(
    OUTPUT_PATHS["candidates"]
)
saved_target = pd.read_parquet(
    OUTPUT_PATHS["selected_target"]
)
saved_horizon_summary = pd.read_csv(
    OUTPUT_PATHS["horizon_summary"]
)

candidate_duplicate_rows = int(
    saved_candidates.duplicated(
        subset=[
            "LOAN IDENTIFIER",
            "HORIZON_MONTHS",
        ]
    ).sum()
)

unexpected_saved_horizons = sorted(
    set(saved_candidates["HORIZON_MONTHS"])
    - set(HORIZONS)
)

selected_target_duplicate_loans = int(
    saved_target["LOAN IDENTIFIER"]
    .duplicated()
    .sum()
)

unexpected_target_values = sorted(
    set(
        saved_target[
            "TARGET_COMPOSITE_CREDIT_EVENT"
        ].dropna()
    )
    - {0, 1}
)

readback_validation = {
    "candidate_rows": len(saved_candidates),
    "candidate_duplicate_loan_horizons": (
        candidate_duplicate_rows
    ),
    "unexpected_saved_horizons": (
        unexpected_saved_horizons
    ),
    "selected_target_rows": len(saved_target),
    "selected_target_duplicate_loans": (
        selected_target_duplicate_loans
    ),
    "unexpected_target_values": (
        unexpected_target_values
    ),
    "selected_horizon_values": sorted(
        saved_target["HORIZON_MONTHS"].unique()
    ),
    "saved_summary_rows": len(
        saved_horizon_summary
    ),
}

if candidate_duplicate_rows > 0:
    raise ValueError(
        "Duplicate loan-horizon rows found after read-back."
    )

if unexpected_saved_horizons:
    raise ValueError(
        "Unexpected horizons found after read-back: "
        f"{unexpected_saved_horizons}"
    )

if selected_target_duplicate_loans > 0:
    raise ValueError(
        "Duplicate selected-target loans found after read-back."
    )

if unexpected_target_values:
    raise ValueError(
        "Unexpected target values found after read-back: "
        f"{unexpected_target_values}"
    )

if set(
    saved_target["HORIZON_MONTHS"].unique()
) != {SELECTED_HORIZON}:
    raise ValueError(
        "Saved target does not contain only the selected horizon."
    )

if len(saved_horizon_summary) != len(HORIZONS):
    raise ValueError(
        "Saved horizon summary has an unexpected row count."
    )

display(
    pd.DataFrame([readback_validation])
)

## 16. Conclusion and Next Stage

This notebook establishes a documented and reproducible 2015 performance-target baseline.

### What is fixed

- The serious-delinquency boundary is numeric status 3 or higher, corresponding to 90+ days delinquent, with `RA` included because it represents REO acquisition.
- Terminal credit events are limited to documented Zero Balance Codes `02`, `03`, `09`, and `15`.
- Voluntary payoff/maturity (`01`), reperforming-loan securitization (`16`), and defect termination (`96`) do not independently create a positive label.
- The outcome clock is anchored to `FIRST PAYMENT DATE`, not modification-sensitive `LOAN AGE`.
- Left-truncated and right-censored loans are not assigned artificial negative labels.
- Overall sample retention and observable-cohort follow-up coverage are reported separately.
- The 95% guard is applied to observable-cohort coverage, while the effect of left-truncation remains visible through the sample-retention measure and exclusion count.
- The 24-month composite target is the primary endpoint only after passing the coverage guard.
- The 12- and 36-month outcomes remain available for sensitivity analysis.

### What remains data-dependent

The executed outputs determine the eligible-loan count, both coverage measures, event counts, event rates, acquisition timing, internal gap frequency, and quarterly outcome variation. Those results must be reviewed before the logic is generalized to 2016-2019.

### Next stage

After the 2015 outputs pass review, move the validated target process into a reusable multi-year pipeline, generate labels for 2016-2019, and then join the labels to the already validated 50-column origination cohorts. Formal drift analysis and Logistic Regression/XGBoost modeling should begin only after those joins and their read-back checks pass.